# Neural Network Experiments

We will train a small MLP with categorical embeddings to see whether a neural network can learn useful patterns beyond gradient boosting.

We will use the same five-fold cross-validation and compare OOF ROC-AUC with our tuned boosting models.

In [6]:
from pathlib import Path
import copy
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

SEED = 42
TARGET = "is_fraud"
N_SPLITS = 5

BATCH_SIZE = 2048
MAX_EPOCHS = 30
PATIENCE = 5
LR = 1e-3
WEIGHT_DECAY = 1e-3

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

root = Path("../playground-series-s6e11")
train_df = pd.read_csv(root / "train.csv")

X = train_df.drop(columns=["id", TARGET])
y = train_df[TARGET].to_numpy(dtype=np.float32)

cat_cols = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

num_cols = X.columns.difference(cat_cols).tolist()

print("Numerical:", len(num_cols))
print("Categorical:", cat_cols)
print("Train shape:", X.shape)

Device: cpu
Numerical: 19
Categorical: ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
Train shape: (699960, 24)


## Feature Preprocessing

We will standardize numerical features and encode categorical values as integer IDs for embedding layers.

The scaler, missing-value statistics, and category mappings will be fitted separately on each training fold.

In [7]:
def prepare_fold(X_train, X_valid):
    # Numerical features
    train_num = X_train[num_cols].astype("float32").copy()
    valid_num = X_valid[num_cols].astype("float32").copy()

    medians = train_num.median()

    train_num = train_num.fillna(medians).fillna(0)
    valid_num = valid_num.fillna(medians).fillna(0)

    scaler = StandardScaler()

    train_num = scaler.fit_transform(train_num).astype("float32")
    valid_num = scaler.transform(valid_num).astype("float32")

    # Categorical embeddings: 0 = unknown / missing
    train_cat = []
    valid_cat = []
    cardinalities = []

    for col in cat_cols:
        tr = X_train[col].astype("string").fillna("__MISSING__")
        va = X_valid[col].astype("string").fillna("__MISSING__")

        categories = tr.unique()
        mapping = {
            value: idx + 1
            for idx, value in enumerate(categories)
        }

        train_cat.append(
            tr.map(mapping).fillna(0).to_numpy(dtype=np.int64)
        )
        valid_cat.append(
            va.map(mapping).fillna(0).to_numpy(dtype=np.int64)
        )

        cardinalities.append(len(categories) + 1)

    train_cat = np.column_stack(train_cat)
    valid_cat = np.column_stack(valid_cat)

    return (
        train_num,
        valid_num,
        train_cat,
        valid_cat,
        cardinalities,
    )

## MLP Architecture

Categorical embeddings and standardized numerical features will be concatenated and passed through three hidden layers.

We will use BatchNorm, ReLU, and Dropout to control overfitting.

In [8]:
class FraudMLP(nn.Module):
    def __init__(self, n_numeric, cardinalities):
        super().__init__()

        embedding_dims = [
            min(8, max(2, size // 2))
            for size in cardinalities
        ]

        self.embeddings = nn.ModuleList([
            nn.Embedding(size, dim)
            for size, dim in zip(cardinalities, embedding_dims)
        ])

        input_dim = n_numeric + sum(embedding_dims)

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1),
        )

    def forward(self, numeric, categorical):
        embeddings = [
            layer(categorical[:, i])
            for i, layer in enumerate(self.embeddings)
        ]

        x = torch.cat([numeric] + embeddings, dim=1)
        return self.network(x).squeeze(1)


def predict_model(model, loader):
    model.eval()
    predictions = []

    with torch.no_grad():
        for numeric, categorical in loader:
            numeric = numeric.to(device)
            categorical = categorical.to(device)

            logits = model(numeric, categorical)
            probabilities = torch.sigmoid(logits)

            predictions.append(probabilities.cpu().numpy())

    return np.concatenate(predictions)

In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

nn_oof = np.zeros(len(X), dtype=np.float32)
fold_results = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), 1):
    print(f"\nFold {fold}")
    set_seed(SEED + fold)

    (
        tr_num, va_num,
        tr_cat, va_cat,
        cardinalities,
    ) = prepare_fold(
        X.iloc[train_idx],
        X.iloc[valid_idx],
    )

    train_loader = DataLoader(
        TensorDataset(
            torch.from_numpy(tr_num),
            torch.from_numpy(tr_cat),
            torch.from_numpy(y[train_idx]),
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    valid_loader = DataLoader(
        TensorDataset(
            torch.from_numpy(va_num),
            torch.from_numpy(va_cat),
        ),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    model = FraudMLP(
        n_numeric=len(num_cols),
        cardinalities=cardinalities,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.BCEWithLogitsLoss()

    best_auc = 0
    best_epoch = 0
    best_state = None
    patience_counter = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0

        for numeric, categorical, target in train_loader:
            numeric = numeric.to(device)
            categorical = categorical.to(device)
            target = target.to(device)

            optimizer.zero_grad()

            logits = model(numeric, categorical)
            loss = criterion(logits, target)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(target)

        valid_pred = predict_model(model, valid_loader)
        valid_auc = roc_auc_score(y[valid_idx], valid_pred)

        print(
            f"Epoch {epoch:02d}: "
            f"loss={total_loss / len(train_idx):.4f}, "
            f"val_auc={valid_auc:.6f}"
        )

        if valid_auc > best_auc + 1e-5:
            best_auc = valid_auc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print("Early stopping")
            break

    model.load_state_dict(best_state)
    nn_oof[valid_idx] = predict_model(model, valid_loader)

    fold_results.append({
        "fold": fold,
        "auc": roc_auc_score(y[valid_idx], nn_oof[valid_idx]),
        "best_epoch": best_epoch,
    })

    print(
        f"Best fold AUC: {best_auc:.6f}, "
        f"best epoch: {best_epoch}"
    )

    del model, train_loader, valid_loader

    if device.type == "cuda":
        torch.cuda.empty_cache()


Fold 1
Epoch 01: loss=0.2287, val_auc=0.922387
Epoch 02: loss=0.0636, val_auc=0.935273
Epoch 03: loss=0.0535, val_auc=0.937541
Epoch 04: loss=0.0513, val_auc=0.938767
Epoch 05: loss=0.0506, val_auc=0.938774
Epoch 06: loss=0.0501, val_auc=0.938983
Epoch 07: loss=0.0495, val_auc=0.940114
Epoch 08: loss=0.0493, val_auc=0.939987
Epoch 09: loss=0.0490, val_auc=0.940027
Epoch 10: loss=0.0488, val_auc=0.940096
Epoch 11: loss=0.0486, val_auc=0.940019
Epoch 12: loss=0.0484, val_auc=0.940182
Epoch 13: loss=0.0485, val_auc=0.940187
Epoch 14: loss=0.0483, val_auc=0.940057
Epoch 15: loss=0.0482, val_auc=0.940014
Epoch 16: loss=0.0481, val_auc=0.940452
Epoch 17: loss=0.0479, val_auc=0.939763
Epoch 18: loss=0.0478, val_auc=0.940267
Epoch 19: loss=0.0478, val_auc=0.939777
Epoch 20: loss=0.0477, val_auc=0.940340
Epoch 21: loss=0.0476, val_auc=0.940373
Early stopping
Best fold AUC: 0.940452, best epoch: 16

Fold 2
Epoch 01: loss=0.2070, val_auc=0.921513
Epoch 02: loss=0.0635, val_auc=0.935241
Epoch 03:

## Results

We will compare the neural network with our tuned boosting models.

Even if the MLP performs worse individually, it may still be useful if its predictions differ from those of LightGBM and XGBoost.

In [10]:
nn_auc = roc_auc_score(y, nn_oof)

print(f"Neural Network OOF AUC: {nn_auc:.6f}")
print(f"LightGBM OOF AUC:       0.947895")
print(f"XGBoost OOF AUC:        0.947502")

display(pd.DataFrame(fold_results))

output_dir = Path("nn_artifacts")
output_dir.mkdir(exist_ok=True)

pd.DataFrame({
    "id": train_df["id"],
    "is_fraud": y.astype(int),
    "oof_pred": nn_oof,
}).to_csv(output_dir / "nn_oof.csv", index=False)

Neural Network OOF AUC: 0.941041
LightGBM OOF AUC:       0.947895
XGBoost OOF AUC:        0.947502


,fold,auc,best_epoch
0,1,0.940452,16
1,2,0.942886,6
2,3,0.946784,15
3,4,0.941096,18
4,5,0.936439,12


## Findings
The MLP achieved an OOF `ROC-AUC` of ***0.941041***, below the tuned `LightGBM` ***(0.947895)*** and `XGBoost` ***(0.947502)***.

The best epoch varied from 6 to 18 across folds. Validation AUC often stopped improving while training loss continued to decrease, suggesting that longer training alone is unlikely to close the gap.

We did not test a neural-network blend in this notebook, so we cannot conclude whether its predictions would help the boosting ensemble. For now, we will not use this MLP in the final submission.

Conclusion: The small embedding-based MLP was a useful alternative architecture to test, but it did not match our tuned boosting models. We will stop this direction here.